In [1]:
import os
import pickle

import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Load data
r_cols = ['user_id', 'movie_id', 'rating', 'unix_timestamp']
ratings_base = pd.read_csv('./ml-100k/ub.base', sep='\t', names=r_cols, encoding='latin-1')
ratings_test = pd.read_csv('./ml-100k/ub.test', sep='\t', names=r_cols, encoding='latin-1')

rate_train = ratings_base.to_numpy()
rate_test = ratings_test.to_numpy()

u_cols =  ['user_id', 'age', 'sex', 'occupation', 'zip_code']
users = pd.read_csv('./ml-100k/u.user', sep='|', names=u_cols, encoding='latin-1')

i_cols = ['movie_id', 'movie_title', 'release_date', 'video_release_date', 'IMDb_URL', 'unknown', 'Action', 'Adventure', 'Animation', "Children's", 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']
items = pd.read_csv('./ml-100k/u.item', sep='|', names=i_cols, encoding='latin-1')

n_users = users.shape[0]
n_items = items.shape[0]


In [2]:
# Utility matrix
Utility = np.zeros((n_users, n_items))
for u, i, r, _ in rate_train:
    Utility[u - 1, i - 1] = r

# Normalize
R_masked = np.ma.array(Utility, mask=(Utility == 0))
user_mean = np.ma.mean(R_masked, axis=1)

Utility_normalized = Utility.copy().astype(float)
for i in range(n_users):
    rated = np.where(Utility[i] > 0)[0]
    Utility_normalized[i, rated] -= user_mean[i]

# Similarity
def compute_similarity(U_norm, mode='item'):
    if mode == 'item':
        sim = cosine_similarity(U_norm.T)
    else:
        sim = cosine_similarity(U_norm)

    np.fill_diagonal(sim, -np.inf)  # remove self
    return sim


 #PREDICT FUNCTION

def predict_ratings_mem(Utility, Utility_normalized, similarity, user_mean, K=30, mode='item'):
    pred = np.zeros(Utility.shape)

    if mode == 'item':
        for u in range(n_users):
            rated_items = np.where(Utility[u] > 0)[0]
            unrated_items = np.where(Utility[u] == 0)[0]

            for j in unrated_items:
                # chỉ lấy item mà user đã rate
                sim_vec = similarity[j, rated_items]

                # chọn top-K neighbors DYNAMIC
                if len(sim_vec) == 0:
                    continue

                top_k_idx = np.argsort(sim_vec)[-K:]
                top_items = rated_items[top_k_idx]
                top_sim = sim_vec[top_k_idx]

                if len(top_sim) == 0:
                    continue

                num = np.dot(top_sim, Utility_normalized[u, top_items])
                denom = np.sum(np.abs(top_sim)) + 1e-8

                pred[u, j] = num / denom
    else: # user-based
        for u in range(n_users):
            unrated_items = np.where(Utility[u] == 0)[0]

            for j in unrated_items:
                users_rated = np.where(Utility[:, j] > 0)[0]

                sim_vec = similarity[u, users_rated]

                if len(sim_vec) == 0:
                    continue

                top_k_idx = np.argsort(sim_vec)[-K:]
                top_users = users_rated[top_k_idx]
                top_sim = sim_vec[top_k_idx]

                num = np.dot(top_sim, Utility_normalized[top_users, j])
                denom = np.sum(np.abs(top_sim)) + 1e-8

                pred[u, j] = num / denom
    return pred

In [3]:
X0 = items.to_numpy()
X_train_counts = X0[:, -19:]

from sklearn.feature_extraction.text import TfidfTransformer
transformer = TfidfTransformer(smooth_idf=True, norm ='l2')
tfidf = transformer.fit_transform(X_train_counts.tolist()).toarray()

def get_items_rated_by_user(rate_matrix, user_id):
    y = rate_matrix[:,0] 
    ids = np.where(y == user_id + 1)[0] 
    item_ids = rate_matrix[ids, 1] - 1  
    scores = rate_matrix[ids, 2]
    return (item_ids, scores)

from sklearn.linear_model import Ridge
def predict_ratings_cb():
    d = tfidf.shape[1]
    W = np.zeros((d, n_users))
    b = np.zeros((1, n_users))

    global_mean = np.mean(rate_train[:, 2])
    for n in range(n_users):    
        ids, scores = get_items_rated_by_user(rate_train, n)
        if len(ids) == 0:
            W[:, n] = np.zeros(d)
            b[0, n] = global_mean
            continue

        model = Ridge(alpha = 1, fit_intercept  = True)
        Xhat = tfidf[ids, :]
        
        model.fit(Xhat, scores) 
        W[:, n] = model.coef_
        b[0, n] = model.intercept_

    Yhat = tfidf.dot(W) + b
    Yhat = np.clip(Yhat, 1, 5)

    return Yhat

In [4]:
def save_similarity_checkpoint(sim, user_mean, Utility_normalized, mode='item'):
    checkpoint = {
        "similarity_matrix": sim,
        "user_mean": user_mean,
        "Utility_normalized": Utility_normalized
    }

    filename = f"{mode}_cf_checkpoint.pkl"

    with open(filename, "wb") as f:
        pickle.dump(checkpoint, f)

    print(f"Saved similarity checkpoint to {filename}")

def load_similarity_checkpoint(mode='item'):
    filename = f"{mode}_cf_checkpoint.pkl"

    if not os.path.exists(filename):
        return None

    with open(filename, "rb") as f:
        checkpoint = pickle.load(f)

    print(f"Loaded similarity checkpoint from {filename}")

    return checkpoint

def save_prediction_matrix(pred, mode='item'):
    filename = f"{mode}_pred.npy"

    np.save(filename, pred)

    print(f"Saved prediction matrix to {filename}")

def load_prediction_matrix(mode='item'):
    filename = f"{mode}_pred.npy"

    if not os.path.exists(filename):
        return None

    pred = np.load(filename)

    print(f"Loaded prediction matrix from {filename}")

    return pred

In [5]:
# RMSE
def compute_RMSE(pred, rate_test):
    mse = 0
    count = 0
    for u, i, r, _ in rate_test:
        mse += (pred[u - 1, i - 1] - r) ** 2
        count += 1
    return np.sqrt(mse / count)

# Recommend
def recommend(user_id, pred, rate_train, K=5):
    user_ratings = pred[user_id - 1]
    rated_items = rate_train[rate_train[:, 0] == user_id][:, 1] - 1
    unrated_items = np.setdiff1d(np.arange(pred.shape[1]), rated_items)

    top_items = np.argsort(user_ratings[unrated_items])[-K:]
    return unrated_items[top_items]

def build_ranking_data(rate_train, rate_test, n_items, n_neg=99):
    user_pos = {}
    user_neg = {}

    all_items = set(range(n_items))

    for u in np.unique(rate_test[:, 0]):
        test_items = rate_test[rate_test[:, 0] == u][:, 1] - 1
        train_items = rate_train[rate_train[:, 0] == u][:, 1] - 1

        if len(test_items) == 0:
            continue

        # lấy 1 positive (leave-one-out)
        pos_item = test_items[0]
        user_pos[u - 1] = pos_item

        # negative sampling
        candidates = list(all_items - set(train_items) - {pos_item})
        if len(candidates) >= n_neg:
            neg_items = np.random.choice(candidates, n_neg, replace=False)
        else:
            neg_items = candidates

        user_neg[u - 1] = neg_items

    return user_pos, user_neg

def precision_at_k(pred, user_pos, user_neg, K=5):
    precisions = []

    for u in user_pos:
        pos_item = user_pos[u]
        neg_items = user_neg[u]

        items = np.append(neg_items, pos_item)

        scores = pred[u, items]
        ranked_items = items[np.argsort(scores)[::-1]]

        top_k = ranked_items[:K]

        hit = int(pos_item in top_k)
        precisions.append(hit / K)

    return np.mean(precisions)

def recall_at_k(pred, user_pos, user_neg, K=5):
    recalls = []

    for u in user_pos:
        pos_item = user_pos[u]
        neg_items = user_neg[u]

        items = np.append(neg_items, pos_item)

        scores = pred[u, items]
        ranked_items = items[np.argsort(scores)[::-1]]

        top_k = ranked_items[:K]

        hit = int(pos_item in top_k)
        recalls.append(hit)

    return np.mean(recalls)

def ndcg_at_k(pred, user_pos, user_neg, K=5):
    ndcgs = []

    for u in user_pos:
        pos_item = user_pos[u]
        neg_items = user_neg[u]

        items = np.append(neg_items, pos_item)

        scores = pred[u, items]
        ranked_items = items[np.argsort(scores)[::-1]]

        # tìm rank của positive item
        rank = np.where(ranked_items == pos_item)[0][0] + 1  # +1 vì rank bắt đầu từ 1

        if rank <= K:
            ndcg = 1 / np.log2(rank + 1)
        else:
            ndcg = 0

        ndcgs.append(ndcg)

    return np.mean(ndcgs)

def mrr_at_k(pred, user_pos, user_neg, K=5):
    mrrs = []

    for u in user_pos:
        pos_item = user_pos[u]
        neg_items = user_neg[u]

        items = np.append(neg_items, pos_item)

        scores = pred[u, items]
        ranked_items = items[np.argsort(scores)[::-1]]

        rank = np.where(ranked_items == pos_item)[0][0] + 1

        if rank <= K:
            mrr = 1 / rank
        else:
            mrr = 0

        mrrs.append(mrr)

    return np.mean(mrrs)

In [6]:
def build_hybrid_prediction(pred_cf, pred_cb, alpha=0.6):
    pred_cb_T = pred_cb.T
    pred_hybrid = alpha * pred_cf + (1 - alpha) * pred_cb_T
    pred_hybrid = np.clip(pred_hybrid, 1, 5)
    
    return pred_hybrid

def evaluate_pred(pred_hybrid, rate_test, user_pos, user_neg, K=5):
    rmse = compute_RMSE(pred_hybrid, rate_test)
    precision = precision_at_k(pred_hybrid, user_pos, user_neg, K)
    recall = recall_at_k(pred_hybrid, user_pos, user_neg, K)
    ndcg = ndcg_at_k(pred_hybrid, user_pos, user_neg, K)
    mrr = mrr_at_k(pred_hybrid, user_pos, user_neg, K)
    
    return {
        'rmse': rmse,
        'precision': precision,
        'recall': recall,
        'ndcg': ndcg,
        'mrr': mrr
    }

In [7]:
def tune_alpha(pred_cf, Yhat_cb, rate_test, user_pos, user_neg, K=5, alphas=None):
    if alphas is None:
        alphas = np.arange(0, 1.05, 0.05)
    
    results = []
    
    for alpha in alphas:
        pred_hybrid = build_hybrid_prediction(pred_cf, Yhat_cb, alpha)
        metrics = evaluate_pred(pred_hybrid, rate_test, user_pos, user_neg, K)
        results.append((alpha, metrics))
        print(f"Alpha = {alpha:.2f}: RMSE = {metrics['rmse']:.4f}, "
              f"Precision@{K} = {metrics['precision']:.4f}, "
              f"Recall@{K} = {metrics['recall']:.4f}, "
              f"NDCG@{K}: {metrics['ndcg']:.4f}, "
              f"MRR@{K}: {metrics['mrr']:.4f}"
              )
    
    best_alpha, best_metrics = min(results, key=lambda x: x[1]['rmse'])
    print(f"\nBest alpha: {best_alpha:.2f} with RMSE = {best_metrics['rmse']:.4f}")
    
    return best_alpha, best_metrics, results

In [8]:
def pred_mem(cf_mode='item', n_neighbors=30, K=5):
    np.random.seed(42)
    print(f"\nRunning {cf_mode}-based CF...")
    
    checkpoint = load_similarity_checkpoint(cf_mode)
    
    if checkpoint is not None:
        sim = checkpoint["similarity_matrix"]

        print("Using saved similarity matrix...")

    else:
        print("Computing similarity matrix...")

        sim = compute_similarity(
            Utility_normalized,
            mode=cf_mode
        )

        save_similarity_checkpoint(
            sim,
            user_mean,
            Utility_normalized,
            cf_mode
        )

    pred = load_prediction_matrix(cf_mode)

    if pred is not None:
        print("Using saved prediction matrix...")

    else:
        print("Computing prediction matrix...")

        Yhat_norm = predict_ratings_mem(Utility,Utility_normalized,sim,user_mean,n_neighbors,mode=cf_mode)

        user_mean_array = np.asarray(user_mean.data)

        pred = Yhat_norm + user_mean_array[:, None]

        pred = np.clip(pred, 1, 5)
        
        save_prediction_matrix(pred, cf_mode)
    
    return pred

In [9]:
def main(K=5):
    pred_item = pred_mem('item')
    
    pred_user = pred_mem('user')

    pred_content = predict_ratings_cb()
    
    user_pos, user_neg = build_ranking_data(rate_train, rate_test, n_items)
    
    # Baseline
    item_metrics = evaluate_pred(pred_item, rate_test, user_pos, user_neg, K)
    print(f"\nItem-based CF Baseline:")
    print(f"RMSE: {item_metrics['rmse']:.4f}")
    print(f"Precision@{K}: {item_metrics['precision']:.4f}")
    print(f"Recall@{K}: {item_metrics['recall']:.4f}")
    print(f"NDCG@{K}: {item_metrics['ndcg']:.4f}")
    print(f"MRR@{K}: {item_metrics['mrr']:.4f}")

    user_metrics = evaluate_pred(pred_user, rate_test, user_pos, user_neg, K)
    print(f"\nUser-based CF Baseline:")
    print(f"RMSE: {user_metrics['rmse']:.4f}")
    print(f"Precision@{K}: {user_metrics['precision']:.4f}")
    print(f"Recall@{K}: {user_metrics['recall']:.4f}")
    print(f"NDCG@{K}: {user_metrics['ndcg']:.4f}")
    print(f"MRR@{K}: {user_metrics['mrr']:.4f}")

    content_metrics = evaluate_pred(pred_content.T, rate_test, user_pos, user_neg, K)
    print(f"\nContent-based CF Baseline:")
    print(f"RMSE: {content_metrics['rmse']:.4f}")
    print(f"Precision@{K}: {content_metrics['precision']:.4f}")
    print(f"Recall@{K}: {content_metrics['recall']:.4f}")
    print(f"NDCG@{K}: {content_metrics['ndcg']:.4f}")
    print(f"MRR@{K}: {content_metrics['mrr']:.4f}")
    
    print("\nHybrid Recommender System:")

    # Tune alpha
    print(f"\nTuning Hybrid (Item-based + Content-based):")
    best_alpha_i, best_metrics_i, _ = tune_alpha(pred_item, pred_content, rate_test, user_pos, user_neg, K)
    
    print("\nCompare Hybrid (Item-based + Content-based) vs Item-based")
    print(f"Item-based CF: RMSE: {item_metrics['rmse']:.4f}")
    print(f"Hybrid (α={best_alpha_i:.2f}): RMSE: {best_metrics_i['rmse']:.4f}")
    improvement = item_metrics['rmse'] - best_metrics_i['rmse']
    improvement_pct = (improvement / item_metrics['rmse']) * 100
    print(f"Improvement: {improvement:.4f} ({improvement_pct:.1f}%)")

    print(f"\nTuning Hybrid (User-based + Content-based):")
    best_alpha_u, best_metrics_u, _ = tune_alpha(pred_user, pred_content, rate_test, user_pos, user_neg, K)
    
    print("\nCompare Hybrid (User-based + Content-based) vs User-based:")
    print(f"User-based CF: RMSE: {user_metrics['rmse']:.4f}")
    print(f"Hybrid (α={best_alpha_u:.2f}): RMSE: {best_metrics_u['rmse']:.4f}")
    improvement = user_metrics['rmse'] - best_metrics_u['rmse']
    improvement_pct = (improvement / user_metrics['rmse']) * 100
    print(f"Improvement: {improvement:.4f} ({improvement_pct:.1f}%)")

if __name__ == "__main__":
    main()


Running item-based CF...
Loaded similarity checkpoint from item_cf_checkpoint.pkl
Using saved similarity matrix...
Loaded prediction matrix from item_pred.npy
Using saved prediction matrix...

Running user-based CF...
Loaded similarity checkpoint from user_cf_checkpoint.pkl
Using saved similarity matrix...
Loaded prediction matrix from user_pred.npy
Using saved prediction matrix...


MemoryError: Unable to allocate 12.1 MiB for an array with shape (1682, 943) and data type float64